Installations

In [ ]:
import torch

TORCH = torch.__version__.split('+')[0]

CUDA = torch.version.cuda.split('+')[0] if torch.cuda.is_available() else 'cpu'

!pip install torch-scatter -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html # for  graph convolution and neighborhood aggregation
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html #for sparse matrix operations
!pip install torch-cluster -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html  #for clustering operations on graphs
!pip install torch-spline-conv -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install torch-geometric

Looking in links: https://data.pyg.org/whl/torch-2.8.0+cpu.html
Looking in links: https://data.pyg.org/whl/torch-2.8.0+cpu.html
Looking in links: https://data.pyg.org/whl/torch-2.8.0+cpu.html
Looking in links: https://data.pyg.org/whl/torch-2.8.0+cpu.html


Imports

In [ ]:
import os
import math
import time
import random
from typing import List, Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool
from torch_geometric.transforms import NormalizeFeatures

BERT Model

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModel
    TRANSFORMERS_AVAILABLE = True
except Exception:
    TRANSFORMERS_AVAILABLE = False

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
BERT_MODEL_NAME = "bert-base-uncased"
USE_BERT = True and TRANSFORMERS_AVAILABLE
BERT_CACHE_DIR = "./bert_cache"

Hyperparameters

In [ ]:
N_EPOCHS = 50
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_DIM = 128
NUM_SAGE_LAYERS = 2
DROPOUT = 0.1
SEED=42
random.seed(SEED)
torch.manual_seed(SEED)

TextEmebedder

In [ ]:
class BertTextEmbedder:
    def __init__(self, model_name: str = BERT_MODEL_NAME, cache_dir: str = BERT_CACHE_DIR, device=DEVICE):
        if not TRANSFORMERS_AVAILABLE:
            raise RuntimeError("transformers library is not available. Install with `pip install transformers`")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()
        os.makedirs(cache_dir, exist_ok=True)
        self.cache_dir = cache_dir
        self.device = device

    def _cache_path(self, text: str) -> str:
        import hashlib
        h = hashlib.sha256(text.encode("utf-8")).hexdigest()
        return os.path.join(self.cache_dir, f"{h}.pt")

    @torch.no_grad()
    def embed(self, text: str) -> torch.Tensor:
        """Return 1D tensor (embedding vector)
        """
        cache_path = self._cache_path(text)
        if os.path.exists(cache_path):
            return torch.load(cache_path)
        toks = self.tokenizer(text, truncation=True, padding=True, return_tensors="pt")
        toks = {k: v.to(self.device) for k, v in toks.items()}
        out = self.model(**toks)
        cls = out.last_hidden_state[:, 0, :].squeeze(0).detach().cpu()
        torch.save(cls, cache_path)
        return cls

GNN Model (GraphSAGE)

In [ ]:
class GNNEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = HIDDEN_DIM, num_layers: int = NUM_SAGE_LAYERS, dropout: float = DROPOUT):
        super().__init__()
        assert num_layers >= 1
        self.convs = nn.ModuleList() #  for storing layers
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x

class FakeNewsGNN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = HIDDEN_DIM, num_layers: int = NUM_SAGE_LAYERS, classifier_hidden: int = 64, dropout: float = DROPOUT):
        super().__init__()
        self.encoder = GNNEncoder(in_dim, hidden_dim, num_layers, dropout)
        # After global pooling -> classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, classifier_hidden),
            nn.BatchNorm1d(classifier_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden, 1)
        )

    def forward(self, data: Data):
        x = data.x
        edge_index = data.edge_index
        batch = data.batch
        node_emb = self.encoder(x, edge_index)
        pooled = global_mean_pool(node_emb, batch)
        out = self.classifier(pooled).squeeze(-1)
        return out

Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    losses = []
    all_preds = []
    all_labels = []

    for batch in loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()

        out = model(batch)
        squeezed_out = out.squeeze(1) if out.dim() > 1 else out

        labels = batch.y.view(-1).to(DEVICE).float()

        loss = loss_fn(squeezed_out, labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        probs = torch.sigmoid(squeezed_out).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_preds.extend(list(preds))
        all_labels.extend(list(labels.detach().cpu().numpy().astype(int)))

    if len(all_labels) == 0:
        return math.nan, {}

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    return float(sum(losses) / len(losses)), {"acc": acc, "prec": prec, "rec": rec, "f1": f1}


Evaluation Loop

In [ ]:
def eval_epoch(model, loader, loss_fn):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            out = model(batch)

            squeezed_out = out.squeeze(1) if out.dim() > 1 else out

            labels = batch.y.view(-1).to(DEVICE).float()

            loss = loss_fn(squeezed_out, labels)
            losses.append(loss.item())

            probs = torch.sigmoid(squeezed_out).detach().cpu().numpy()
            preds = (probs >= 0.5).astype(int)

            all_preds.extend(list(preds))
            all_labels.extend(list(labels.detach().cpu().numpy().astype(int)))

    if len(all_labels) == 0:
        return math.nan, {}

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    return float(sum(losses) / len(losses)), {"acc": acc, "prec": prec, "rec": rec, "f1": f1}

Main Run

In [ ]:
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader

def main():
    print(f"Device: {DEVICE}")
    print("Loading UPFD dataset (Politifact, BERT features)...")

    # Load dataset
    train_dataset = UPFD(root='data/UPFD', name='politifact', feature='bert', split='train')
    val_dataset   = UPFD(root='data/UPFD', name='politifact', feature='bert', split='val')
    test_dataset  = UPFD(root='data/UPFD', name='politifact', feature='bert', split='test')
    #Normalization
    train_dataset.transform = NormalizeFeatures()
    val_dataset.transform = NormalizeFeatures()
    test_dataset.transform = NormalizeFeatures()
    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    in_dim = train_dataset.num_node_features
    print("Feature dim:", in_dim)

    # Model setup
    model = FakeNewsGNN(in_dim=in_dim).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_val_f1 = -1.0
    best_model_path = "best_fake_news_gnn.pt"

    # Training loop
    for epoch in range(1, N_EPOCHS + 1):
        train_loss, train_metrics = train_epoch(model, train_loader, optimizer, loss_fn)
        val_loss, val_metrics = eval_epoch(model, val_loader, loss_fn)

        print(f"Epoch {epoch:02d} | "
              f"train_loss={train_loss:.4f} acc={train_metrics.get('acc',0):.4f} f1={train_metrics.get('f1',0):.4f} | "
              f"val_loss={val_loss:.4f} acc={val_metrics.get('acc',0):.4f} f1={val_metrics.get('f1',0):.4f}")

        if val_metrics.get('f1', 0) > best_val_f1:
            best_val_f1 = val_metrics.get('f1', 0)
            torch.save(model.state_dict(), best_model_path)

    # Testing
    model.load_state_dict(torch.load(best_model_path))
    test_loss, test_metrics = eval_epoch(model, test_loader, loss_fn)
    print("Test results:", test_loss, test_metrics)


if __name__ == "__main__":
    main()


Device: cpu
Loading UPFD dataset (Politifact, BERT features)...
Feature dim: 768
Epoch 01 | train_loss=0.7037 acc=0.4355 f1=0.3137 | val_loss=0.6971 acc=0.4194 f1=0.0000
Epoch 02 | train_loss=0.6879 acc=0.5323 f1=0.5085 | val_loss=0.6975 acc=0.4194 f1=0.0000
Epoch 03 | train_loss=0.6916 acc=0.4355 f1=0.3396 | val_loss=0.6979 acc=0.4194 f1=0.0000
Epoch 04 | train_loss=0.7074 acc=0.4355 f1=0.3636 | val_loss=0.6983 acc=0.4194 f1=0.0000
Epoch 05 | train_loss=0.6939 acc=0.5000 f1=0.3673 | val_loss=0.6986 acc=0.4194 f1=0.0000
Epoch 06 | train_loss=0.6747 acc=0.5484 f1=0.4400 | val_loss=0.6989 acc=0.4194 f1=0.0000
Epoch 07 | train_loss=0.7006 acc=0.5000 f1=0.3404 | val_loss=0.6991 acc=0.4194 f1=0.0000
Epoch 08 | train_loss=0.6716 acc=0.6290 f1=0.5106 | val_loss=0.6993 acc=0.4194 f1=0.0000
Epoch 09 | train_loss=0.7170 acc=0.3710 f1=0.2041 | val_loss=0.6995 acc=0.4194 f1=0.0000
Epoch 10 | train_loss=0.6716 acc=0.6129 f1=0.5200 | val_loss=0.6997 acc=0.4194 f1=0.0000
Epoch 11 | train_loss=0.6758 

gossipcop aletenate dataset

In [ ]:
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader

In [ ]:
train_dataset = UPFD(root='data/UPFD', name='politifact', feature='bert', split='train')
val_dataset   = UPFD(root='data/UPFD', name='politifact', feature='bert', split='val')
test_dataset  = UPFD(root='data/UPFD', name='politifact', feature='bert', split='test')